In [ ]:
import numpy as np
import string 

import pandas as pd

sanitizer = lambda x: x.replace('‘', '').replace('…', '').replace(";", "").replace("—", " ").translate(str.maketrans('', '', string.punctuation)).strip().lower() #patchwork ass sanitization function that removes leading spaces, puncutation and replaces - with a space
clean_script_list = np.loadtxt("../data/hobbit1.csv", delimiter=',', usecols = 1,  skiprows = 1, dtype=str, converters = sanitizer, encoding="utf-8")

df = pd.read_csv("../data/hobbit1.csv", delimiter = ','); #hopefully not cheating
clean_script_list = [sanitizer(line) for line in df["line"].tolist()]; 

tokenized_script = [sentence.split() for sentence in clean_script_list] #split into 2d array of sentences split into words

vocab = sorted(set(word for sentence in tokenized_script for word in sentence)) #get a sorted set of unique words
word_to_index = {word: idx for idx, word in enumerate(vocab)} #initialize a dict, word corresponds to a value index

def one_hot_generator(word, vocab_size): 
    one_hot = np.zeros(vocab_size)
    one_hot[word_to_index[word]] = 1 
    return one_hot



['60', 'a', 'aaahh', 'aahhh', 'abandoned', 'about', 'absurd', 'academic', 'accepted', 'accustomed', 'ack', 'across', 'acts', 'actually', 'addled', 'adol', 'advantage', 'adventure', 'adventures', 'advice', 'ae', 'afraid', 'after', 'again', 'against', 'age', 'ago', 'agree', 'agreeable', 'agreed', 'ah', 'ahead', 'ahh', 'ahhh', 'ain’t', 'air', 'airborne', 'alarm', 'ale', 'alive', 'all', 'allegiance', 'allow', 'alone', 'along', 'already', 'alright', 'also', 'although', 'always', 'am', 'ambushed', 'among', 'amongst', 'amount', 'amusing', 'an', 'ancient', 'and', 'angmar', 'angmar’s', 'animal', 'annam', 'another', 'answer', 'answerable', 'answered', 'antique', 'any', 'anyone', 'anything', 'anywhere', 'apology', 'appear', 'appearances', 'appears', 'appreciation', 'approve', 'are', 'argh', 'arm', 'armchair', 'armed', 'armor', 'arms', 'army', 'around', 'arrangements', 'artifact', 'as', 'ash', 'ask', 'asked', 'asleep', 'assessing', 'astride', 'at', 'attached', 'attacked', 'attacks', 'autumn', 'ave

In [ ]:
#need a user specified window-size, hardcode this to 3 for now 
#CBOW IMPLEMENTATION
WINDOW_SIZE = 3

def CBOW_generate_training_data(tokenized_sen, word_dict, window_size = 3):
    X_train = []
    y_train = []

    V = len(word_dict)

    for sen in tokenized_sen: #at this point, we assume our data is a 2d array, of tokenized sentences, so iterate through those sen
        for i, target in enumerate(sen): #slide through whole sentece, each word is a target word, no loop backs here for now

            aggregated_cv = np.zeros(V)

            
            start = max(0, i - window_size) #the start is capped to the first 
            end = min(len(sen), i + window_size) #end is capped to last element

            context = [sen[j] for j in sen[start:end + 1] if j != i] #comprehension list, iterate through all words in our sublist window, keep target word out.
            for word in context: 
                aggregated_cv[word_dict[word]] += 1.0 

            if len(context) > 0:
                aggregated_cv /= len(context)

            y_train.append(one_hot_generator(target, V)) #and append to context list to data matrix, and corresponding one hot target word rep in target list.
            X_train.append(aggregated_cv)

    return np.array(X_train).T, np.array(y_train).T
#np.reshape(np.array(X_train), (-1, len(vocab))), np.reshape(np.array(y_train), (-1, 1))

X_train, y_train = CBOW_generate_training_data(tokenized_script, word_to_index, 2)


In [ ]:
def softmax(X):
    X_max = np.max(X, axis=0, keepdims=True)
    e_X = np.exp(X - X_max)
    #something about numerical stability will add later if i understand it 
    return e_X / np.sum(e_X, axis = 0, keepdims = True) #dumb luck

def forward_prop(X, W0, W1, b0, b1):

    #X : aggregated onehot input for group of words
    A1 = W0.T @ X + b0 #U: intermediate output, context vectors
    #no activation function on hidden layer output  
    A2 = W1.T @ A1 + b1 #Prediction between vectors 
    Y_predict= softmax(A2) #Y: apply softmax to obtain final prediction, will be plugged into loss
    return A1, A2, Y_predict

def back_prop(X, Y, Y_predict, A1, W1, M):
    dLdb1, dLdb0 = 0, 0
    dLdu = Y_predict - Y # partial deriv dL/du
    dLdW1 = (1 / M) * (A1 @ dLdu.T)
    dLdb1 = (1 / M) * (dLdu @ np.ones((M, 1)))
    dLdW0 = (1 / M) * (X @ (W1 @ dLdu).T)

    return dLdW0, dLdW1, dLdb0, dLdb1
     

def update_model(W0, W1, b0, b1, dLdW0, dLdW1, dLdb0, dLdb1, alpha):
    W1_new = W1 - alpha * dLdW1
    W0_new = W0 - alpha * dLdW0

    b1_new = b1 - alpha * dLdb1 
    b0_new = b0 - alpha * dLdb0
    return W0_new, W1_new, b0_new, b1_new

def print_summary(Y, Y_predict, iteration): 
    loss = -np.mean(np.sum(Y * np.log(Y_predict + 1e-12), axis=0))
    true_val = np.argmax(Y, axis = 0)

    top_10_predict_val = np.argsort(Y_predict, axis = 0)[-10:]
    true_val_index = np.argmax(Y, axis = 0)
    
    correct_in_10 = [true_val_index[i] in top_10_predict_val[:, i] for i in range(Y.shape[1])]
    top_10_accuracy = np.mean(correct_in_10)

    print(f'iteration: {iteration}')
    print(f"loss: {loss}")
    print(f"accuracy: {top_10_accuracy}")

def vanilla_grad_desc(X, Y, W0, W1, b0, b1, alpha = 0.01, epochs = 50):
    M = X.shape[1]
    for i in range(epochs): 
        A1, A2, Y_predict = forward_prop(X, W0, W1, b0, b1)
        dLdW0, dLdW1, dLdb0, dLdb1 = back_prop(X, Y, Y_predict, A1, W1, M)
        
        W0, W1, b0, b1 = update_model(W0, W1, b0, b1, dLdW0, dLdW1, dLdb0, dLdb1, alpha)

        if(i % 5 == 0):
            print_summary(Y, Y_predict, i)
    return W0, W1, b0, b1

def vanilla_stoch_grad_desc(X, Y, W0, W1, b0, b1, alpha = 0.01, epochs = 50): 
    M = X.shape[1]
    for i in range(epochs / M): 
        for j in range(M): 
            X_isolated = X[:, [j]]
            Y_isolated = Y[:, [j]]
    
            A1, A2, Y_predict = forward_prop(X_isolated, W0, W1, b0, b1)
            dLdW0, dLdW1, dLdb0, dLdb1 = back_prop(X_isolated, Y_isolated, Y_predict, A1, W1, M)
            W0, W1, b0, b1 = update_model(W0, W1, b0, b1, dLdW0, dLdW1, dLdb0, dLdb1, alpha)

    return W0, W1, b0, b1

def negative_sampling_grad_desc(X, Y, W0, W1, b0, b1, alpha = 0.01, epochs = 50):
    



EMBEDDING_DIM = int(np.sqrt(len(vocab))) #hard coded for now

def init_vals(): 
    W0 = np.random.randn(len(vocab), EMBEDDING_DIM) #matrix to map to trait nodes
    W1 = np.random.randn(EMBEDDING_DIM, len(vocab)) #
    b0 = np.zeros((EMBEDDING_DIM, 1))
    b1 = np.zeros((len(vocab), 1))
    return W0, W1, b0, b1 

#for now we'll stick with vanilla gradient descent, will figure out stochastic later.

W0, W1, b0, b1 = init_vals()
W0, W1, b0, b1  = vanilla_grad_desc(X_train, y_train, W0, W1, b0, b1, 0.1, 10000)



In [ ]:
while(True): 
    context_words = []
    while(True):
        word = ''
        while(word not in word_to_index): 
            word = input()
            if((word in word_to_index) or (word == "END")): 
                break
            print("not in vocab")

        if(word == "END"):
            break
        
        print(f'added word: {word}')
        context_words.append(word)

    aggregated_cv = np.zeros((len(vocab), 1))

    for word in context_words: 
        aggregated_cv[word_to_index[word]] += 1.0 

    A0, A1, prediction = forward_prop(aggregated_cv, W0, W1, b0, b1)
    print(vocab[np.argmax(prediction.flatten(), axis = 0)])